# Week 2, Day 2: Gradio

Small user interfaces in front of Gemini and a local Llama 3.2 model.

In [ ]:
import os
import json
from urllib.parse import urljoin
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from scraper import fetch_website_contents, fetch_website_links

load_dotenv(override=True)
api_key = os.getenv("GEMINI_API_KEY")
print("GEMINI_API_KEY found" if api_key else "GEMINI_API_KEY not set")

GEMINI = "gemini-3.5-flash-lite"
LLAMA = "llama3.2"

gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=api_key)
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

MODELS = {
    "Gemini": (gemini, GEMINI),
    "Ollama": (ollama, LLAMA),
}

HELPFUL = "You are a helpful assistant that responds in markdown without code blocks"

## A function behind a text box

In [ ]:
def shout(text):
    return text.upper()


gr.Interface(
    fn=shout,
    title="Shout",
    inputs=gr.Textbox(label="Your message:", lines=7),
    outputs=gr.Textbox(label="Response:", lines=8),
    examples=["hello", "howdy"],
    flagging_mode="never",
).launch()

## The same interface, calling Gemini

In [ ]:
def message_gemini(prompt):
    response = gemini.chat.completions.create(
        model=GEMINI,
        messages=[
            {"role": "system", "content": HELPFUL},
            {"role": "user", "content": prompt},
        ],
    )
    return response.choices[0].message.content


message_gemini("What is today's date?")

In [ ]:
gr.Interface(
    fn=message_gemini,
    title="Gemini",
    inputs=gr.Textbox(label="Your message:", info="Enter a message for Gemini", lines=7),
    outputs=gr.Markdown(label="Response:"),
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
    ],
    flagging_mode="never",
).launch()

## Stream the reply, and pick Gemini or Llama

In [ ]:
def stream_chat(client, model, prompt, system):
    stream = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        stream=True,
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result


def stream_model(prompt, model):
    client, model_name = MODELS[model]
    yield from stream_chat(client, model_name, prompt, HELPFUL)


gr.Interface(
    fn=stream_model,
    title="LLMs",
    inputs=[
        gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7),
        gr.Dropdown(["Gemini", "Ollama"], label="Select model", value="Gemini"),
    ],
    outputs=gr.Markdown(label="Response:"),
    examples=[
        ["Explain the Transformer architecture to a layperson", "Gemini"],
        ["Explain the Transformer architecture to an aspiring AI engineer", "Ollama"],
    ],
    flagging_mode="never",
).launch()

## Company brochure generator

Give it a company name and landing-page URL. The selected model picks the relevant links (about, careers, and similar), reads those pages, and streams a short brochure.

In [ ]:
LINK_SYSTEM = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

BROCHURE = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


def select_relevant_links(url, client, model_name):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    user_prompt += "\n".join(fetch_website_links(url))
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": LINK_SYSTEM},
            {"role": "user", "content": user_prompt},
        ],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)


def brochure_prompt(company_name, url, client, model_name):
    prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.

## Landing Page:

{fetch_website_contents(url)}
## Relevant Links:
"""
    for link in select_relevant_links(url, client, model_name)["links"]:
        page = urljoin(url, link["url"])
        prompt += f"\n\n### Link: {link['type']}\n"
        prompt += fetch_website_contents(page)
    return prompt[:5_000]


def stream_brochure(company_name, url, model):
    yield "Choosing the pages worth including..."
    client, model_name = MODELS[model]
    prompt = brochure_prompt(company_name, url, client, model_name)
    yield from stream_chat(client, model_name, prompt, BROCHURE)


gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator",
    inputs=[
        gr.Textbox(label="Company name:"),
        gr.Textbox(label="Landing page URL including http:// or https://"),
        gr.Dropdown(["Gemini", "Ollama"], label="Select model", value="Gemini"),
    ],
    outputs=gr.Markdown(label="Response:"),
    examples=[
        ["Hugging Face", "https://huggingface.co", "Gemini"],
        ["Edward Donner", "https://edwarddonner.com", "Ollama"],
    ],
    flagging_mode="never",
).launch()